# GP2 GBA1 MAVE Project

### Single variant association analysis

### Set Up

In [ ]:
%pip install openpyxl
#restart

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for DataFrame functionality
import pandas as pd

import subprocess

# Numpy for basics
import numpy as np

# Use pathlib for file path manipulation
import pathlib

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

import re

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

In [ ]:
## GP2 v10.0
REL12_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release12')
!ls -hal {REL12_PATH}/clinical_data

In [ ]:
# Set paths
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/extended_clinical_data_release12_vwb.csv')
CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/master_key_release12_final_vwb.csv')
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/r12_extended_clinical_data_vwb.csv')
RELATED_DATA_PATH = pathlib.Path(REL12_PATH, 'meta_data/related_samples/')
RAW_GENO_PATH = pathlib.Path(REL12_PATH, 'raw_genotypes')
IMPUTED_GENO_PATH = pathlib.Path(REL12_PATH, 'imputed_genotypes')
WGS_GENO = '/home/jupyter/workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/plink'
WGS_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/pcs')
RELATED = '/home/jupyter/workspace/gp2_tier2_eu_release12/meta_data/related_samples'

### Install Packages

In [ ]:
%%capture
%%bash

# Install plink 1.9
cd /home/jupyter/tools/
if test -e /home/jupyter/tools/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter/tools/

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x /home/jupyter/tools/plink1.9

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/tools/
if test -e /home/jupyter/tools/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/tools/

wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20250122.zip

unzip -o plink2_linux_x86_64_20250122.zip

fi

In [ ]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/tools/plink2

In [ ]:
%%bash

# Install ANNOVAR: We are adding the download link after registration on the annovar website
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then

echo "annovar is already installed in /home/jupyter/notebooks"
else
echo "annovar is not installed"
cd /home/jupyter/tools/

wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz

tar xvfz annovar.latest.tar.gz

fi

In [ ]:
%%bash
#%%capture


# Install ANNOVAR: Download resources for annotation
cd /home/jupyter/tools/annovar/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20140902 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb cytoBand humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ensGene humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar exac03 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp147 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ #latest version of dbNSFP
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad211_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar ljb26_all humandb/

In [ ]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/tools/rvtests
cd /home/jupyter/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz

tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
%%bash

cd /home/jupyter/tools/rvtests
#make
chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

In [ ]:
%%bash

#install htslib   (bgzip and tabix)
if test -e /home/jupyter/tools/htslib/bin; then

echo "htslib is already installed"
else
echo "htslib is not installed"

mkdir /home/jupyter/tools/htslib
cd /home/jupyter/tools/htslib

wget https://github.com/samtools/htslib/releases/download/1.21/htslib-1.21.tar.bz2

tar -jxvf htslib-1.21.tar.bz2

cd /home/jupyter/tools/htslib/htslib-1.21
./configure --prefix=/home/jupyter/tools/htslib/
make
make install

fi

In [ ]:
# Add htslib directory to the PATH
htslib_path = '/home/jupyter/tools/htslib/bin/'
# Check if the path is already in the PATH variable
if htslib_path not in os.environ['PATH']:
    # If not, add the path to the PATH variable
    os.environ['PATH'] += f':{htslib_path}'
    print(f"Path added to PATH: {htslib_path}")
else:
    # If the path is already in PATH, print a message
    print(f"The path is already in PATH: {htslib_path}")

### Association analysis

#### Create a .txt file with the genetic variant information (include all variants you are interested in)
Use the following format: chr:pos:ref:alt  

In [ ]:
! grep "chr4:89771534:CT:C" /home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11.pvar

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" \
  --keep "${KEEP}" \
  --snp "${VAR}" \
  --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
  --adjust \
  --pheno "${COV}" --pheno-name PHENO \
  --covar "${COV}" --covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
  --covar-variance-standardize \
  --out "${OUT}"

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "${PFX}" \
  --keep "${KEEP}" \
  --pheno "${COV}" --pheno-name PHENO \
  --snp "${VAR}" \
  --freq case-control counts \
  --out "${WORK_DIR_ANC}/AFR_${VAR//:/_}.freq"

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"
TAG="AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "${PFX}" \
  --keep "${KEEP}" \
  --pheno "${COV}" --pheno-name PHENO \
  --snp "${VAR}" \
  --export A \
  --out "${WORK_DIR_ANC}/${TAG}.recodeA"

In [ ]:
PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"
TAG = f"AFR_{VAR.replace(':','_')}"

raw_path = f"{WORK_DIR_ANC}/{TAG}.recodeA.raw"
recode = pd.read_csv(raw_path, delim_whitespace=True)

# Find the single variant column (everything after the 6 fixed columns)
fixed = {"FID","IID","PAT","MAT","SEX","PHENOTYPE"}
variant_cols = [c for c in recode.columns if c not in fixed]
if len(variant_cols) != 1:
    raise ValueError(f"Expected 1 variant column, found {len(variant_cols)}: {variant_cols}")
vcol = variant_cols[0]

cases = recode[recode["PHENOTYPE"] == 2].copy()
controls = recode[recode["PHENOTYPE"] == 1].copy()

# Treat missing as NA (PLINK sometimes uses -9)
for df in (cases, controls):
    df.loc[df[vcol] < 0, vcol] = pd.NA

def carrier_stats(df):
    n = df.shape[0]
    n_nonmiss = df[vcol].notna().sum()
    hom = (df[vcol] == 0).sum()
    het = (df[vcol] == 1).sum()
    carriers = hom + het
    carrier_freq = carriers / n_nonmiss if n_nonmiss else float("nan")
    return n, n_nonmiss, hom, het, carriers, carrier_freq

n_case, n_case_nm, hom_case, het_case, car_case, cf_case = carrier_stats(cases)
n_ctrl, n_ctrl_nm, hom_ctrl, het_ctrl, car_ctrl, cf_ctrl = carrier_stats(controls)

out = pd.DataFrame([{
    "Variant_ID": VAR,
    "Genotype_col": vcol,
    "Cases_N": n_case,
    "Cases_nonmissing": n_case_nm,
    "Cases_hom": int(hom_case),
    "Cases_het": int(het_case),
    "Cases_carriers": int(car_case),
    "Cases_carrier_freq": cf_case,
    "Controls_N": n_ctrl,
    "Controls_nonmissing": n_ctrl_nm,
    "Controls_hom": int(hom_ctrl),
    "Controls_het": int(het_ctrl),
    "Controls_carriers": int(car_ctrl),
    "Controls_carrier_freq": cf_ctrl,
}])

out_path = f"{WORK_DIR_ANC}/{TAG}.carrier_freq.txt"
out.to_csv(out_path, sep="\t", index=False)
out


In [ ]:
glm_path = f"{WORK_DIR_ANC}/{TAG}.PHENO.glm.logistic.hybrid"  # adjust if yours ends with .hybrid
assoc = pd.read_csv(glm_path, delim_whitespace=True)
assoc_add = assoc[assoc["TEST"] == "ADD"].copy()

carrier = pd.read_csv(f"{WORK_DIR_ANC}/{TAG}.carrier_freq.txt", sep="\t")

merged = carrier.merge(
    assoc_add[["ID","A1","A1_FREQ","OBS_CT","OR","LOG(OR)_SE","Z_STAT","P"]],
    left_on="Variant_ID", right_on="ID", how="left"
)

merged.to_csv(f"{WORK_DIR_ANC}/{TAG}.glm_plus_carrierfreq.txt", sep="\t", index=False)
merged


In [ ]:
import numpy as np
x = recode["chr4:89771534:CT:C_CT"].replace(-9, np.nan)
(x.mean()/2)


### Run with all AFR samples

### Create a new covariate file with GP2 data

In [ ]:
# Let's load the master key
key = pd.read_csv("/home/jupyter/workspace/gp2_tier2_eu_release11/clinical_data/master_key_release11_final_vwb.csv")
print(key.shape)
key.head()

In [ ]:
# Subsetting to keep only a few columns
key = key[['GP2ID', 'study', 'baseline_GP2_phenotype', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'wgs_label', 'study_type', 'family_history_for_qc']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype':'phenotype',
                                     'biological_sex_for_qc':'SEX',
                                     'age_at_sample_collection':'AGE',
                                     'age_of_onset':'AAO',
                                     'wgs_label': 'label'}, inplace = True)
key

In [ ]:
ANC = "AFR"

# Paths you already have in your notebook/env:
# WORK_DIR, RELATED, WGS_PATH, key  (key is your master key dataframe)

WORK_DIR_ANC = f"{WORK_DIR}/{ANC}"
os.makedirs(WORK_DIR_ANC, exist_ok=True)

print(f"WORKING ON: {ANC}")

# 1) Subset to ancestry + study (site acronym)
dfk = key[(key["label"] == ANC)]
dfk = dfk.reset_index(drop=True)

print(f"After ancestry filter: {dfk.shape}")

# 2) Remove related individuals (your existing logic: remove IID1 only)
related_df = pd.read_csv(f"{RELATED}/{ANC}_release11_vwb.related")
related_list = set(related_df["IID1"].astype(str))
dfk["IID"] = dfk["IID"].astype(str)
dfk = dfk[~dfk["IID"].isin(related_list)].copy()

print(f"After relatedness filter: {dfk.shape}")

# 3) Phenotype -> PLINK coding (1/2, -9 missing)
dfk["PHENO"] = -9

dfk.loc[dfk["phenotype"].isin(["Control", "Population Control"]), "PHENO"] = 1
dfk.loc[dfk["phenotype"] == "PD", "PHENO"] = 2

# Affected_PD = case only if study_type == Monogenic
mask = (dfk["phenotype"] == "Affected_PD") & (dfk["study_type"] == "Monogenic")
dfk.loc[mask, "PHENO"] = 2

dfk["PHENO"] = dfk["PHENO"].astype("Int64")
print("PHENO counts:\n", dfk["PHENO"].value_counts(dropna=False).to_string())

# 4) PCs (AFR-only eigenvec)
# If your eigenvec is whitespace-delimited with header like: FID IID PC1 PC2 ...
pcs = pd.read_csv(f"{WGS_PATH}/{ANC}/{ANC}_release11.eigenvec", sep=r"\s+")

# Ensure IID column name is IID (2nd column is usually IID)
pcs = pcs.rename(columns={pcs.columns[1]: "IID"})
pcs["IID"] = pcs["IID"].astype(str)

pcs = pcs[["IID", "PC1", "PC2", "PC3", "PC4", "PC5"]].copy()
print(f"PCs: {pcs.shape}")

# 5) SEX recode to PLINK convention (Male=1, Female=2)
print("SEX original:\n", dfk["SEX"].value_counts(dropna=False).to_string())

sex_mapping = {"Female": 2, "Male": 1}
dfk["SEX"] = dfk["SEX"].map(sex_mapping).astype("Int64")

print("SEX recoded:\n", dfk["SEX"].value_counts(dropna=False).to_string())

# 6) Merge PCs + metadata
df = pd.merge(pcs, dfk, on="IID", how="inner")
print(f"Merged (PCs ∩ key): {df.shape}")

# 7) Build covariate file (PLINK-friendly + your RVtests columns)
df["FID"] = 0
df["FATID"] = 0
df["MATID"] = 0

final_df = df[["IID", "IID", "FATID", "MATID", "SEX", "AGE", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5"]].copy()
final_df.columns = ["FID", "IID", "FATID", "MATID", "SEX", "AGE", "PHENO", "PC1", "PC2", "PC3", "PC4", "PC5"]

# Missingness checks (optional but useful)
print("PD missing AGE:", final_df[(final_df["PHENO"] == 2) & (final_df["AGE"].isna())].shape[0])
print("CTRL missing AGE:", final_df[(final_df["PHENO"] == 1) & (final_df["AGE"].isna())].shape[0])
print("PC1 missing fraction:", final_df["PC1"].isna().mean())

# 8) Write keep + covariate files
keep_path = f"{WORK_DIR_ANC}/{ANC}_all_AFR.samplestoKeep"
cov_path  = f"{WORK_DIR_ANC}/{ANC}_all_AFR_covariate_file.txt"

final_df[["FID", "IID"]].to_csv(keep_path, sep="\t", index=False, header=None)
final_df.to_csv(cov_path, sep="\t", na_rep="NA", index=False)

print("Wrote keep file:", keep_path)
print("Wrote covariate file:", cov_path)


In [ ]:
WORK_DIR_ANC = f'{WORK_DIR}/{ANC}'
# Read your covariate file
covar = pd.read_csv(f'{WORK_DIR_ANC}/{ANC}_all_AFR_covariate_file.txt', sep='\t')

# Define the covariates used in your --covar-name argument
covars_this_run = ['AGE','SEX','PC1','PC2','PC3','PC4','PC5']

# 1. Check how many missing values per covariate
print(covar[covars_this_run].isna().sum().sort_values(ascending=False))

# 2. Check the exact column names (detect trailing spaces, case issues)
print(list(covar.columns))

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR_all_AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_all_AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" \
  --keep "${KEEP}" \
  --snp "${VAR}" \
  --glm hide-covar no-x-sex firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+err \
  --adjust \
  --pheno "${COV}" --pheno-name PHENO \
  --covar "${COV}" --covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
  --covar-variance-standardize \
  --out "${OUT}_all_AFR"

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR_all_AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_all_AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_all_AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "${PFX}" \
  --keep "${KEEP}" \
  --pheno "${COV}" --pheno-name PHENO \
  --snp "${VAR}" \
  --freq case-control counts \
  --out "${WORK_DIR_ANC}/AFR_all_AFR_${VAR//:/_}.freq"

In [ ]:
%%bash

PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR_all_AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_all_AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_all_AFR_${VAR//:/_}"
TAG="AFR_${VAR//:/_}"

/home/jupyter/tools/plink2 \
  --pfile "${PFX}" \
  --keep "${KEEP}" \
  --pheno "${COV}" --pheno-name PHENO \
  --snp "${VAR}" \
  --export A \
  --out "${WORK_DIR_ANC}/${TAG}_all_AFR.recodeA"

In [ ]:
PFX="/home/jupyter/workspace/gp2_tier2_eu_release11/wgs/deepvariant_joint_calling/plink/AFR/chr4_AFR_release11" # points to .pgen/.pvar/.psam
WORK_DIR_ANC="/home/jupyter/workspace/ws_files/R11_LoF_assoc/AFR"   # where keep + cov files live

KEEP="${WORK_DIR_ANC}/AFR_all_AFR.samplestoKeep"
COV="${WORK_DIR_ANC}/AFR_all-AFR_covariate_file.txt"

VAR="chr4:89771534:CT:C"
OUT="${WORK_DIR_ANC}/AFR_${VAR//:/_}"
TAG = f"AFR_{VAR.replace(':','_')}"

raw_path = f"{WORK_DIR_ANC}/{TAG}_all_AFR.recodeA.raw"
recode = pd.read_csv(raw_path, delim_whitespace=True)

# Find the single variant column (everything after the 6 fixed columns)
fixed = {"FID","IID","PAT","MAT","SEX","PHENOTYPE"}
variant_cols = [c for c in recode.columns if c not in fixed]
if len(variant_cols) != 1:
    raise ValueError(f"Expected 1 variant column, found {len(variant_cols)}: {variant_cols}")
vcol = variant_cols[0]

cases = recode[recode["PHENOTYPE"] == 2].copy()
controls = recode[recode["PHENOTYPE"] == 1].copy()

# Treat missing as NA (PLINK sometimes uses -9)
for df in (cases, controls):
    df.loc[df[vcol] < 0, vcol] = pd.NA

def carrier_stats(df):
    n = df.shape[0]
    n_nonmiss = df[vcol].notna().sum()
    hom = (df[vcol] == 0).sum()
    het = (df[vcol] == 1).sum()
    carriers = hom + het
    carrier_freq = carriers / n_nonmiss if n_nonmiss else float("nan")
    return n, n_nonmiss, hom, het, carriers, carrier_freq

n_case, n_case_nm, hom_case, het_case, car_case, cf_case = carrier_stats(cases)
n_ctrl, n_ctrl_nm, hom_ctrl, het_ctrl, car_ctrl, cf_ctrl = carrier_stats(controls)

out = pd.DataFrame([{
    "Variant_ID": VAR,
    "Genotype_col": vcol,
    "Cases_N": n_case,
    "Cases_nonmissing": n_case_nm,
    "Cases_hom": int(hom_case),
    "Cases_het": int(het_case),
    "Cases_carriers": int(car_case),
    "Cases_carrier_freq": cf_case,
    "Controls_N": n_ctrl,
    "Controls_nonmissing": n_ctrl_nm,
    "Controls_hom": int(hom_ctrl),
    "Controls_het": int(het_ctrl),
    "Controls_carriers": int(car_ctrl),
    "Controls_carrier_freq": cf_ctrl,
}])

out_path = f"{WORK_DIR_ANC}/{TAG}_all_AFR.carrier_freq.txt"
out.to_csv(out_path, sep="\t", index=False)
out


In [ ]:
glm_path = f"{WORK_DIR_ANC}/{TAG}_all_AFR.PHENO.glm.logistic.hybrid"  # adjust if yours ends with .hybrid
assoc = pd.read_csv(glm_path, delim_whitespace=True)
assoc_add = assoc[assoc["TEST"] == "ADD"].copy()

carrier = pd.read_csv(f"{WORK_DIR_ANC}/{TAG}_all_AFR.carrier_freq.txt", sep="\t")

merged = carrier.merge(
    assoc_add[["ID","A1","A1_FREQ","OBS_CT","OR","LOG(OR)_SE","Z_STAT","P"]],
    left_on="Variant_ID", right_on="ID", how="left"
)

merged.to_csv(f"{WORK_DIR_ANC}/{TAG}_all_AFR.glm_plus_carrierfreq.txt", sep="\t", index=False)
merged


In [ ]:
%%bash

WORKDIR="/home/jupyter/workspace/ws_files/R12_GBA1_MAVE"

awk -F'\t' 'NR>1 {print $1":"$2":"$3":"$4}' \
"${WORKDIR}/variants.txt" \
> "${WORKDIR}/variant_list.txt"

head "${WORKDIR}/variant_list.txt"

In [ ]:
%%bash

BASE_PFILE_DIR="/home/jupyter/workspace/gp2_tier2_eu_release12/wgs/dragen_joint_calling/plink"
BASE_WORK_DIR="/home/jupyter/workspace/ws_files/R12_GBA1_MAVE"
VARLIST="${BASE_WORK_DIR}/variant_list.txt"

ancestries=(AAC AFR AJ AMR CAH CAS EAS EUR FIN MDE SAS)
COVAR_NAMES="AGE SEX PC1 PC2 PC3 PC4 PC5"
CHR=1

for ANC in "${ancestries[@]}"; do

  echo "============================================================"
  echo "WORKING ON: ${ANC}"

  WORK_DIR_ANC="${BASE_WORK_DIR}/${ANC}"
  KEEP="${WORK_DIR_ANC}/${ANC}_all_${ANC}.samplestoKeep"
  COV="${WORK_DIR_ANC}/${ANC}_all_${ANC}_covariate_file.txt"
  OUTDIR="${WORK_DIR_ANC}/variant_freq_assoc"
  mkdir -p "${OUTDIR}"

  PFX="${BASE_PFILE_DIR}/${ANC}/chr${CHR}_${ANC}_release12"
  OUT="${OUTDIR}/${ANC}_chr${CHR}"

  if [[ ! -f "${PFX}.pgen" || ! -f "${KEEP}" || ! -f "${COV}" ]]; then
    echo "WARNING: Missing pgen/keep/cov file for ${ANC}, skipping."
    continue
  fi

  PRESENT="${OUTDIR}/${ANC}_chr${CHR}_variants_present.txt"

  if [[ -f "${PFX}.pvar" ]]; then
    awk 'NR==FNR {v[$1]=1; next} ($3 in v) {print $3}' "${VARLIST}" "${PFX}.pvar" > "${PRESENT}"
  elif [[ -f "${PFX}.pvar.gz" ]]; then
    zcat "${PFX}.pvar.gz" | awk 'NR==FNR {v[$1]=1; next} ($3 in v) {print $3}' "${VARLIST}" - > "${PRESENT}"
  else
    echo "WARNING: No pvar found for ${ANC}, skipping."
    continue
  fi

  if [[ ! -s "${PRESENT}" ]]; then
    echo "No requested variants found for ${ANC}"
    continue
  fi

  echo "Cases-only frequencies for ${ANC}"
  /home/jupyter/tools/plink2 \
    --pfile "${PFX}" \
    --keep "${KEEP}" \
    --pheno "${COV}" --pheno-name PHENO \
    --keep-if "PHENO == 2" \
    --extract "${PRESENT}" \
    --freq counts \
    --out "${OUT}_cases"

  echo "Controls-only frequencies for ${ANC}"
  /home/jupyter/tools/plink2 \
    --pfile "${PFX}" \
    --keep "${KEEP}" \
    --pheno "${COV}" --pheno-name PHENO \
    --keep-if "PHENO == 1" \
    --extract "${PRESENT}" \
    --freq counts \
    --out "${OUT}_controls"

  echo "Association / OR for ${ANC}"
  /home/jupyter/tools/plink2 \
    --pfile "${PFX}" \
    --keep "${KEEP}" \
    --pheno "${COV}" --pheno-name PHENO \
    --covar "${COV}" --covar-name ${COVAR_NAMES} \
    --covar-variance-standardize \
    --extract "${PRESENT}" \
    --glm firth-fallback hide-covar \
    --ci 0.95 \
    --out "${OUT}_assoc"

done

In [ ]:
import os
import glob
import pandas as pd

BASE_WORK_DIR = "/home/jupyter/workspace/ws_files/R12_GBA1_MAVE"
ancestries = ['AAC', 'AFR', 'AJ', 'AMR', 'CAH', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS']

all_results = []

for ANC in ancestries:
    outdir = f"{BASE_WORK_DIR}/{ANC}/variant_freq_assoc"

    cases_file = f"{outdir}/{ANC}_chr1_cases.acount"
    controls_file = f"{outdir}/{ANC}_chr1_controls.acount"
    assoc_files = glob.glob(f"{outdir}/{ANC}_chr1_assoc.PHENO.glm.*")

    if not os.path.exists(cases_file):
        print(f"Missing cases file for {ANC}")
        continue

    if not os.path.exists(controls_file):
        print(f"Missing controls file for {ANC}")
        continue

    cases = pd.read_csv(cases_file, sep=r"\s+")
    controls = pd.read_csv(controls_file, sep=r"\s+")

    cases = cases.rename(columns={
        "ID": "variant",
        "ALT_CTS": "PD_AC",
        "OBS_CT": "PD_N"
    })

    controls = controls.rename(columns={
        "ID": "variant",
        "ALT_CTS": "CTRL_AC",
        "OBS_CT": "CTRL_N"
    })

    cases["PD_AN"] = cases["PD_N"] * 2
    controls["CTRL_AN"] = controls["CTRL_N"] * 2

    cases["PD_AF"] = cases["PD_AC"] / cases["PD_AN"]
    controls["CTRL_AF"] = controls["CTRL_AC"] / controls["CTRL_AN"]

    cases = cases[["variant", "REF", "ALT", "PD_AC", "PD_AF", "PD_AN", "PD_N"]]
    controls = controls[["variant", "REF", "ALT", "CTRL_AC", "CTRL_AF", "CTRL_AN", "CTRL_N"]]

    merged = pd.merge(
        cases,
        controls,
        on=["variant", "REF", "ALT"],
        how="outer"
    )

    if len(assoc_files) > 0:
        assoc = pd.read_csv(assoc_files[0], sep=r"\s+")

        if "TEST" in assoc.columns:
            assoc = assoc[assoc["TEST"] == "ADD"].copy()

        assoc = assoc.rename(columns={
            "ID": "variant",
            "A1": "TESTED_ALLELE"
        })

        assoc_cols = [
            "variant", "TESTED_ALLELE", "OR", "L95", "U95",
            "LOG(OR)_SE", "P", "OBS_CT"
        ]
        assoc_cols = [c for c in assoc_cols if c in assoc.columns]

        assoc = assoc[assoc_cols].copy()

        merged = pd.merge(
            merged,
            assoc,
            on="variant",
            how="outer"
        )
    else:
        print(f"Missing assoc file for {ANC}")

    merged.insert(0, "ANCESTRY", ANC)
    all_results.append(merged)

final = pd.concat(all_results, ignore_index=True)

out_path = f"{BASE_WORK_DIR}/GBA1_MAVE_chr1_case_control_freq_OR.tsv"
final.to_csv(out_path, sep="\t", index=False)

print("Wrote:", out_path)
final.head()